# Expérience 2 — Notre propre tokenizer maghrébin

**Question :** un tokenizer BPE entraîné sur de la darija découpe-t-il mieux la darija que les tokenizers des grands modèles ? À partir de quelle taille de vocabulaire ? Et doit-il apprendre aussi l'**arabizi** ?

**Ce que l'expérience 1 nous a appris :**
- les tokenizers récents (Llama 3, Gemma, Qwen) gèrent correctement la darija en alphabet arabe ;
- les tokenizers spécialisés (DarijaBERT) sont très bons en alphabet arabe mais **mauvais en arabizi** (2,3 fois plus de tokens) ;
- en arabizi, 11 à 16 % des tokens contiennent un chiffre : les `3`, `7`, `9` sont souvent séparés du mot.

**Ce qu'on fait varier :**

| Facteur | Valeurs |
| --- | --- |
| Taille du vocabulaire | 4 000, 8 000, 16 000, 32 000 |
| Écritures apprises | alphabet arabe seul / arabe **+ arabizi** |
| Découpage avant BPE | standard (comme GPT-2 : chiffres séparés des lettres) / **chiffres gardés dans les mots** (`m3a`, `mti99en`) |

Soit 4 × 2 × 2 = 16 tokenizers.

**Données :**
- **entraînement** : DODa (90 % des phrases), en attendant le corpus du rôle Données ;
- **test**, jamais vu pendant l'entraînement : les 10 % restants de DODa (arabe et arabizi) et FLORES+ (darija marocaine et tunisienne).

**Résultat (à remplir à la fin) :**

## 0. Mise en place

Le CPU suffit : entraîner un tokenizer est rapide. Mêmes secrets que pour l'expérience 1 (`HF_TOKEN`, et `GITHUB_TOKEN` si le dépôt est privé).

In [ ]:
import os, sys

REPO = "Ninessss/darija-llm"
BRANCH = "exp2/tokenizer-bpe"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    if not os.path.exists("/content/darija-llm"):
        try:
            gh = userdata.get("GITHUB_TOKEN")
            url = f"https://{gh}@github.com/{REPO}.git"
        except Exception:
            url = f"https://github.com/{REPO}.git"
        !git clone -q {url} /content/darija-llm
    %cd /content/darija-llm
    !git fetch -q && git checkout -q {BRANCH} && git pull -q
    !echo "Commit utilisé : $(git rev-parse --short HEAD)"
    !pip install -q transformers tokenizers datasets sentencepiece protobuf pandas matplotlib
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

sys.path.insert(0, os.getcwd())
os.makedirs("results/figures", exist_ok=True)
print("Dossier de travail :", os.getcwd())

In [ ]:
from huggingface_hub import login

try:
    token = userdata.get("HF_TOKEN") if IN_COLAB else os.environ.get("HF_TOKEN")
    login(token=token)
    print("Connecté à Hugging Face")
except Exception as e:
    print("Pas de token Hugging Face :", e)

## 1. Les données

### 1a. DODa, découpé en entraînement / test

Chaque phrase DODa existe en alphabet arabe et en arabizi. On garde **90 %** des phrases pour entraîner et **10 %** pour tester, avec une graine fixe (`SEED = 42`) : le découpage est le même à chaque lancement.

In [ ]:
import numpy as np
import pandas as pd

SEED = 42
DODA_URL = "https://raw.githubusercontent.com/darija-open-dataset/dataset/main/sentences/sentences.csv"

doda = (pd.read_csv(DODA_URL)[["darija", "darija_ar"]]
        .rename(columns={"darija": "arabizi", "darija_ar": "arabe"})
        .dropna().astype(str))
doda = doda[(doda["arabizi"].str.strip() != "") & (doda["arabe"].str.strip() != "")].drop_duplicates()

doda = doda.sample(frac=1, random_state=SEED).reset_index(drop=True)
n_test = len(doda) // 10
doda_test, doda_train = doda.iloc[:n_test], doda.iloc[n_test:]
print(f"DODa : {len(doda_train)} phrases d'entraînement, {len(doda_test)} de test")

### 1b. Corpus du groupe (optionnel)

Quand le rôle Données aura produit le corpus de pré-entraînement, indiquez son chemin ici : il sera **ajouté** aux phrases d'entraînement. Le fichier attendu est un Parquet avec au moins une colonne `text` et, si possible, `split` et `script`.

In [ ]:
EXTRA_CORPUS = "data/processed/pretrain.parquet"   # à adapter

extra_train = []
if os.path.exists(EXTRA_CORPUS):
    extra = pd.read_parquet(EXTRA_CORPUS)
    if "split" in extra.columns:
        extra = extra[extra["split"] == "train"]
    extra_train = extra["text"].dropna().astype(str).tolist()
    print(f"Corpus du groupe ajouté : {len(extra_train)} textes")
else:
    print("Pas de corpus du groupe pour l'instant : entraînement sur DODa seul.")

### 1c. Les jeux de test

FLORES+ n'est **jamais** utilisé pour l'entraînement : c'est un test sur des textes formels et longs, très différents de DODa. La darija tunisienne vérifie si un tokenizer appris surtout sur du marocain se généralise à un dialecte voisin.

In [ ]:
from datasets import load_dataset

def load_flores(code):
    parts = [load_dataset("openlanguagedata/flores_plus", code, split=s).to_pandas()["text"] for s in ["dev", "devtest"]]
    return pd.concat(parts).astype(str).tolist()

TESTS = {
    "FLORES darija ma. (arabe)":  load_flores("ary_Arab"),
    "FLORES darija tn. (arabe)":  load_flores("aeb_Arab"),
    "DODa test (arabe)":          doda_test["arabe"].tolist(),
    "DODa test (arabizi)":        doda_test["arabizi"].tolist(),
}
for name, texts in TESTS.items():
    print(f"{name:<28} {len(texts):>6} phrases")

## 2. Entraîner un tokenizer BPE

Le **BPE** (*Byte Pair Encoding*) part des lettres seules, puis fusionne **la paire la plus fréquente** du corpus, encore et encore, jusqu'à atteindre la taille de vocabulaire voulue. Les morceaux fréquents (`ال`, `ch`, `kan`…) deviennent ainsi des tokens.

On le construit comme GPT-2 et Llama, en **byte-level** : aucun caractère n'est jamais inconnu.

**Le découpage avant BPE** (*pré-tokenisation*) coupe d'abord le texte en mots. Le BPE ne peut jamais fusionner au-delà de ces frontières.
- **standard** : la règle de GPT-2, qui sépare toujours les chiffres des lettres. `mti99en` devient `mti` | `99` | `en` avant même le BPE.
- **chiffres dans les mots** : lettres, chiffres et signes diacritiques restent ensemble. `mti99en` reste un seul mot, et le BPE peut apprendre `99en` ou `3a`.

In [ ]:
from tokenizers import Tokenizer, Regex, models, trainers, pre_tokenizers, decoders, normalizers
from transformers import PreTrainedTokenizerFast

SPECIAL = ["<|endoftext|>", "<pad>"]

def make_pretokenizer(digits_in_words):
    if not digits_in_words:
        return pre_tokenizers.ByteLevel(add_prefix_space=False)          # règle GPT-2
    words = pre_tokenizers.Split(Regex(r" ?[\p{L}\p{M}\p{N}]+| ?[^\s\p{L}\p{M}\p{N}]+|\s+"), behavior="isolated")
    return pre_tokenizers.Sequence([words, pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)])

def train_bpe(texts, vocab_size, digits_in_words):
    tok = Tokenizer(models.BPE())
    tok.normalizer = normalizers.NFC()
    tok.pre_tokenizer = make_pretokenizer(digits_in_words)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size, min_frequency=2, special_tokens=SPECIAL,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)
    tok.train_from_iterator(texts, trainer=trainer)
    return PreTrainedTokenizerFast(tokenizer_object=tok, eos_token="<|endoftext|>", pad_token="<pad>")

# Démonstration sur un mot d'arabizi
for d in [False, True]:
    t = train_bpe(doda_train["arabizi"].tolist()[:5000], 2000, d)
    print("chiffres dans les mots" if d else "standard              ", "->", t.tokenize("ana mti99en m3ak"))

## 3. Les 16 tokenizers

Le vocabulaire réel peut être **plus petit** que demandé : `min_frequency=2` impose qu'une fusion apparaisse au moins deux fois. Avec DODa seul, on n'atteindra sans doute pas 32 000. La colonne `vocab_reel` le montre, et c'est un résultat en soi : il faut plus de données pour justifier un grand vocabulaire.

In [ ]:
import itertools, time

VOCAB_SIZES = [4000, 8000, 16000, 32000]
CORPORA = {
    "arabe":           doda_train["arabe"].tolist() + extra_train,
    "arabe+arabizi":   doda_train["arabe"].tolist() + doda_train["arabizi"].tolist() + extra_train,
}

ours = {}
for (corpus, texts), v, digits in itertools.product(CORPORA.items(), VOCAB_SIZES, [False, True]):
    name = f"BPE {v//1000}k | {corpus} | {'chiffres-mots' if digits else 'standard'}"
    t0 = time.time()
    ours[name] = train_bpe(texts, v, digits)
    print(f"{name:<45} vocab réel {len(ours[name]):>6}   ({time.time()-t0:.0f} s)")

## 4. Les tokenizers de référence

Les mêmes que dans l'expérience 1, sauf mBERT et DziriBERT, pour garder les figures lisibles.

In [ ]:
from transformers import AutoTokenizer

REF_IDS = {
    "GPT-2":      "gpt2",
    "Llama 3":    "meta-llama/Meta-Llama-3-8B",
    "Qwen 2.5":   "Qwen/Qwen2.5-0.5B",
    "Gemma 2":    "google/gemma-2-2b",
    "DarijaBERT": "SI2M-Lab/DarijaBERT",
}
refs = {}
for name, hf_id in REF_IDS.items():
    try:
        refs[name] = AutoTokenizer.from_pretrained(hf_id)
        print(f"OK      {name:<11} (vocabulaire : {len(refs[name]):,})")
    except Exception as e:
        print(f"ÉCHEC   {name:<11} -> {type(e).__name__}: {str(e)[:100]}")

## 5. Les mesures

Pour chaque tokenizer et chaque jeu de test :
- `tokens_total` : nombre total de tokens ;
- `fertilite` : tokens par mot ;
- `vs_llama3` : tokens du tokenizer / tokens de Llama 3 **sur les mêmes phrases**. **En dessous de 1, on fait mieux que Llama 3** ;
- `taux_octets` : part des tokens inconnus ou réduits à des morceaux de lettres.

> **Pourquoi pas la parité par rapport au français ?** Nos tokenizers n'ont jamais vu de français : ils le découperaient très mal, et la parité paraîtrait artificiellement excellente (c'est le piège de la référence, vu à l'expérience 1). Ici, on compare directement **le nombre de tokens sur les mêmes phrases de darija**, ce qui est juste quel que soit le tokenizer.

In [ ]:
from tqdm.auto import tqdm
from src.metrics import is_broken_token

def token_counts(tok, texts):
    ids = tok(texts, add_special_tokens=False)["input_ids"]
    return np.array([len(x) for x in ids]), ids

def broken_rate(tok, ids):
    flat = [i for x in ids for i in x]
    broken = {i for i in set(flat) if is_broken_token(tok, i)}
    return float(np.mean([i in broken for i in flat])) if flat else 0.0

all_toks = {**refs, **ours}
llama_counts = {t: token_counts(refs["Llama 3"], texts)[0] for t, texts in TESTS.items()} if "Llama 3" in refs else None

rows = []
for tok_name, tok in tqdm(all_toks.items()):
    for test_name, texts in TESTS.items():
        counts, ids = token_counts(tok, texts)
        n_words = sum(len(t.split()) for t in texts)
        row = {
            "tokenizer": tok_name,
            "famille": "référence" if tok_name in refs else "nous",
            "vocab_reel": len(tok),
            "test": test_name,
            "tokens_total": int(counts.sum()),
            "fertilite": counts.sum() / n_words,
            "taux_octets": broken_rate(tok, ids),
        }
        if llama_counts is not None:
            row["vs_llama3"] = float(np.mean(counts / np.maximum(llama_counts[test_name], 1)))
        rows.append(row)

results = pd.DataFrame(rows)
parts = results["tokenizer"].str.extract(r"BPE (\d+)k \| (.+) \| (.+)")
results["taille_k"] = pd.to_numeric(parts[0])
results["corpus"] = parts[1]
results["pretok"] = parts[2]
results.to_csv("results/exp2_bpe.csv", index=False)
print(len(results), "lignes enregistrées dans results/exp2_bpe.csv")

In [ ]:
# Tableau principal : fertilité par jeu de test (plus c'est bas, mieux c'est)
fert = results.pivot_table(index="tokenizer", columns="test", values="fertilite")[list(TESTS)]
fert["moyenne"] = fert.mean(axis=1)
fert.sort_values("moyenne").round(2)

In [ ]:
# Comparaison directe avec Llama 3 (< 1 = mieux que Llama 3)
if "vs_llama3" in results:
    display(results.pivot_table(index="tokenizer", columns="test", values="vs_llama3")[list(TESTS)]
            .sort_values("DODa test (arabizi)").round(2))

## 6. Figure : la fertilité selon la taille du vocabulaire

Un graphique par jeu de test. Les courbes sont nos tokenizers, les lignes horizontales les références. Le point intéressant est le **coude** de la courbe : la taille à partir de laquelle agrandir le vocabulaire ne fait presque plus gagner.

In [ ]:
import matplotlib.pyplot as plt

STYLE = {
    ("arabe", "standard"):              dict(color="#4C78A8", ls="--", marker="o"),
    ("arabe", "chiffres-mots"):         dict(color="#4C78A8", ls="-",  marker="s"),
    ("arabe+arabizi", "standard"):      dict(color="#F58518", ls="--", marker="o"),
    ("arabe+arabizi", "chiffres-mots"): dict(color="#F58518", ls="-",  marker="s"),
}
REF_COLORS = {"GPT-2": "#9e9e9e", "Llama 3": "#54A24B", "Qwen 2.5": "#B279A2", "Gemma 2": "#E45756", "DarijaBERT": "#72B7B2"}

fig, axes = plt.subplots(1, len(TESTS), figsize=(5 * len(TESTS), 4.5), sharey=False)
for ax, test_name in zip(axes, TESTS):
    sub = results[results.test == test_name]
    for (corpus, pretok), st in STYLE.items():
        line = sub[(sub.corpus == corpus) & (sub.pretok == pretok)].sort_values("taille_k")
        ax.plot(line.taille_k, line.fertilite, label=f"nous : {corpus}, {pretok}", **st)
    for ref, c in REF_COLORS.items():
        r = sub[sub.tokenizer == ref]
        if len(r):
            ax.axhline(r.fertilite.iloc[0], color=c, lw=1, ls=":")
            ax.text(32.5, r.fertilite.iloc[0], ref, color=c, fontsize=8, va="center")
    ax.set_xscale("log", base=2)
    ax.set_xticks([4, 8, 16, 32], ["4k", "8k", "16k", "32k"])
    ax.set_title(test_name, fontsize=10)
    ax.set_xlabel("taille du vocabulaire demandée")
axes[0].set_ylabel("fertilité (tokens par mot)")
axes[0].legend(fontsize=7, loc="upper right")
fig.suptitle("Expérience 2 — notre BPE contre les tokenizers de référence")
fig.tight_layout()
fig.savefig("results/figures/exp2_fertilite_vocab.png", dpi=200)
plt.show()

## 7. Choisir le tokenizer final

Règle fixée **à l'avance**, pour ne pas choisir « à l'œil » :
1. on garde la variante `arabe+arabizi | chiffres-mots`, sauf si les résultats montrent qu'une autre est meilleure partout ;
2. on prend la **plus petite** taille dont la fertilité moyenne est à moins de **5 %** de la meilleure taille.

Un vocabulaire plus petit rend le mini-GPT de l'expérience 3 plus léger et plus rapide à entraîner.

In [ ]:
CHOSEN_CORPUS, CHOSEN_PRETOK, TOLERANCE = "arabe+arabizi", "chiffres-mots", 0.05

cand = (results[(results.corpus == CHOSEN_CORPUS) & (results.pretok == CHOSEN_PRETOK)]
        .groupby(["tokenizer", "taille_k"])["fertilite"].mean().reset_index().sort_values("taille_k"))
best = cand["fertilite"].min()
final_row = cand[cand["fertilite"] <= best * (1 + TOLERANCE)].iloc[0]
FINAL_NAME = final_row["tokenizer"]
final_tok = ours[FINAL_NAME]
display(cand.assign(ecart_vs_meilleur=lambda d: (d.fertilite / best - 1).map("{:+.1%}".format)))
print("Tokenizer retenu :", FINAL_NAME, "| vocabulaire réel :", len(final_tok))

OUT_DIR = f"tokenizers/darija-bpe-{int(final_row['taille_k'])}k"
final_tok.save_pretrained(OUT_DIR)
print("Enregistré dans", OUT_DIR, ":", os.listdir(OUT_DIR))

## 8. Voir la différence

La même phrase découpée par Llama 3, DarijaBERT et notre tokenizer, dans les deux écritures.

In [ ]:
from src.viz import show_tokens

show = {k: v for k, v in refs.items() if k in ["Llama 3", "DarijaBERT"]}
show["Notre BPE"] = final_tok
i = 0  # changez l'indice pour voir d'autres phrases
for col in ["arabe", "arabizi"]:
    print(f"--- {col} : {doda_test[col].iloc[i]}")
    show_tokens(doda_test[col].iloc[i], show)

## 9. Publier le tokenizer (optionnel)

Le tokenizer est un petit dossier (quelques Mo). Deux façons de le garder :
- **dans le dépôt** : téléchargez le dossier `tokenizers/…` et ajoutez-le avec un commit (l'expérience 3 en aura besoin) ;
- **sur Hugging Face**, comme prévu dans les livrables : il faut un token avec le droit **Write**. Mettez `PUBLISH = True` et choisissez le nom.

In [ ]:
PUBLISH = False
HF_REPO = "Ammar9831/darija-bpe"   # ou le nom de l'organisation du groupe

if PUBLISH:
    final_tok.push_to_hub(HF_REPO, private=True)
    print("Publié sur https://huggingface.co/" + HF_REPO)

## 10. À noter dans `results/journal.md`

In [ ]:
import datetime, subprocess

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
today = datetime.date.today().isoformat()
sel = results[results.tokenizer.isin([FINAL_NAME, "Llama 3", "DarijaBERT"])]
for _, r in sel.iterrows():
    print(f"| {today} | 2 | {r.tokenizer}, {r.test} | fertilité | {r.fertilite:.2f} | {SEED} | {commit} | Ammar |")

## 11. Suite

- [ ] Lire les résultats : le coude de la courbe, l'effet de l'arabizi, l'effet des chiffres dans les mots.
- [ ] Relancer avec le **corpus du groupe** dès qu'il est prêt (section 1b).
- [ ] Tokenizer **commun** contre tokenizers **par dialecte** : nécessite des données tunisiennes et algériennes d'entraînement.
- [ ] Ajouter le tokenizer retenu au dépôt (dossier `tokenizers/`) pour l'expérience 3.